# ⚡ Hourly Building Energy Demand Prediction
**Chulalongkorn Building · 7 Floors · 2018–2019**

---

## Overview
This notebook predicts hourly electricity demand for a 7-floor university building using a LightGBM model with advanced time-series feature engineering and Optuna hyperparameter tuning.

## Dataset
This project uses a dataset from Kaggle:
https://www.kaggle.com/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data
Please download the dataset and place it in the `data/` folder before running this notebook.

## Improvements over naive baseline
- **Richer features:** lag_48, lag_168, rolling_48, rolling_168, month, is_weekend, hour_sin/cos (cyclic encoding)
- **Better imputation:** forward-fill instead of fillna(0) to preserve sensor continuity
- **Systematic tuning:** Optuna with TimeSeriesSplit inner loop (no data leakage)
- **Model comparison:** Linear Regression → LightGBM baseline → LightGBM tuned

## Table of Contents
0. Dependencies
1. Imports
2. Load & Merge CSVs
3. EDA
4. Missing Values Rates
5. Drop High-Missing Columns
6. Fill Missing Values
7. Parse Date & Sort
8. Extract & Aggregate Power Meters
9. Total Demand Distribution
10. Feature Engineering
11. Correlation Heatmap
12. Train / Test Split
13. LightGBM Baseline Model
14. Hyperparameter Tuning (Optuna)
15. Optuna Visualisations
16. Train Final Tuned Model
17. **Model Comparison (LR → LightGBM baseline → LightGBM tuned)**
18. Metrics Comparison Table
19. Cross-Validation
20. CV Plot
21. Actual vs Predicted (Full Test Set)
22. Actual vs Predicted (One Week)
23. Residual Analysis
24. Feature Importance
25. Demand Patterns
26. Final Summary
27. Save Model

In [ ]:
# ── 0. Dependencies ────────────────────────────────────────────────────────────
!pip install pandas numpy lightgbm scikit-learn matplotlib seaborn optuna --quiet

In [ ]:
# ── 1. Imports ─────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8')
print('All imports OK')

In [ ]:
# ── 2. Load & Merge CSVs ───────────────────────────────────────────────────────
folder_path = 'data'
all_files   = os.listdir(folder_path)

df_list = []
for file in all_files:
    if file.endswith('.csv'):
        fp = os.path.join(folder_path, file)
        tmp = pd.read_csv(fp)
        tmp['source_file'] = file
        df_list.append(tmp)

df_merged = pd.concat(df_list, ignore_index=True)
print('Merged shape:', df_merged.shape)
df_merged.head()

In [ ]:
# ── 3. Quick EDA ───────────────────────────────────────────────────────────────
df_merged.info()
df_merged.describe()

In [ ]:
# ── 4. Drop High-Missing Columns (>70 %) ──────────────────────────────────────
missing_pct   = df_merged.isnull().mean() * 100
cols_to_drop  = missing_pct[missing_pct > 70].index

df_clean      = df_merged.drop(columns=cols_to_drop)

print(f'Dropped {len(cols_to_drop)} columns. Remaining shape: {df_clean.shape}')

In [ ]:
# ── 5. Missing Values Rates ──────────────────────────────────────
import matplotlib.pyplot as plt
missing_pct = df_merged.isnull().mean() * 100
missing_pct_sorted = missing_pct.sort_values(ascending=False)
plt.figure(figsize=(14, 5))
missing_pct_sorted.plot(kind='bar', color='steelblue', edgecolor='white')
plt.axhline(70, color='red', linestyle='--', label='70% threshold')
plt.title('Missing Value Rate per Column')
plt.ylabel('Missing (%)')
plt.xlabel('Column')
plt.xticks(rotation=90, fontsize=7)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Fill Remaining Missing Values ──────────────────────────────────────────
# IMPROVEMENT: forward-fill (then back-fill as safety net) instead of fillna(0),
# which would corrupt sensor readings with spurious zeros.
df_clean = df_clean.ffill().bfill()
print('Remaining missing values:', df_clean.isnull().sum().sum())

In [ ]:
# ── 7. Parse Date & Sort ───────────────────────────────────────────────────────
df = df_clean.copy()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.set_index('Date').sort_index()
df.head()

In [ ]:
# ── 8. Extract & Aggregate Power Meters ───────────────────────────────────────
df_power = df.loc[:, df.columns.str.contains('kW')].copy()
df_power['total_demand'] = df_power.sum(axis=1)

df_hourly = df_power['total_demand'].resample('h').mean()

df_hourly.plot(figsize=(14, 4))
plt.title('Total Energy Demand (Hourly)')
plt.ylabel('kW')
plt.tight_layout()
plt.show()
print(df_hourly.head())

In [ ]:
# ── 9. Total Demand Distribution ───────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.hist(df_hourly.dropna(), bins=60, color='steelblue', edgecolor='white')
plt.axvline(df_hourly.mean(), color='red', linestyle='--', label=f'Mean = {df_hourly.mean():.1f} kW')
plt.title('Distribution of Hourly Total Demand')
plt.xlabel('Total Demand (kW)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Feature Engineering ─────────────────────────────────────────────────────

df_feat = df_hourly.reset_index().copy()
df_feat.columns = ['Date', 'total_demand']
df_feat = df_feat.dropna()

# Calendar features
df_feat['hour']       = df_feat['Date'].dt.hour
df_feat['weekday']    = df_feat['Date'].dt.weekday
df_feat['month']      = df_feat['Date'].dt.month
df_feat['is_weekend'] = (df_feat['weekday'] >= 5).astype(int)

# Cyclic encoding for hour (avoids cliff between 23 → 0)
df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24)
df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24)

# Lag features
df_feat['lag_1']   = df_feat['total_demand'].shift(1)
df_feat['lag_24']  = df_feat['total_demand'].shift(24)
df_feat['lag_48']  = df_feat['total_demand'].shift(48)
df_feat['lag_168'] = df_feat['total_demand'].shift(168)   # same hour last week

# Rolling statistics
df_feat['rolling_24']  = df_feat['total_demand'].rolling(24).mean()
df_feat['rolling_48']  = df_feat['total_demand'].rolling(48).mean()
df_feat['rolling_168'] = df_feat['total_demand'].rolling(168).mean()

df_feat = df_feat.set_index('Date').dropna()
print('Feature matrix shape:', df_feat.shape)
df_feat.head()

In [ ]:
# ── 11. Correlation Heatmap ─────────────────────────────────────────────────────
plt.figure(figsize=(12, 8))
sns.heatmap(df_feat.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.3)
plt.title('Correlation Heatmap — All Features')
plt.tight_layout()
plt.show()

In [ ]:
# ── 12. Train / Test Split (Time-Based) ────────────────────────────────────────
FEATURES = [
    'hour', 'weekday', 'month', 'is_weekend',
    'hour_sin', 'hour_cos',
    'lag_1', 'lag_24', 'lag_48', 'lag_168',
    'rolling_24', 'rolling_48', 'rolling_168',
]
TARGET = 'total_demand'

train = df_feat.loc[: '2019-06-30'].copy()
test  = df_feat.loc['2019-07-01':].copy()

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── 13. Baseline Model (original hyperparameters) ──────────────────────────────
baseline = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

rmse_b = np.sqrt(mean_squared_error(y_test, pred_baseline))
mae_b  = mean_absolute_error(y_test, pred_baseline)
r2_b   = r2_score(y_test, pred_baseline)
mape_b = 100 * np.mean(np.abs(pred_baseline - y_test) / np.where(y_test == 0, 1, y_test))

print('─── Baseline ───')
print(f'RMSE : {rmse_b:.3f}')
print(f'MAE  : {mae_b:.3f}')
print(f'MAPE : {mape_b:.2f}%')
print(f'R²   : {r2_b:.4f}')

In [ ]:
# ── 14. Hyperparameter Tuning with Optuna ──────────────────────────────────────

tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 150),
        'max_depth'        : trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'     : 42,
        'verbose'          : -1,
    }
    scores = []
    for tr_idx, val_idx in tscv.split(X_train):
        Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
        scores.append(np.sqrt(mean_squared_error(yval, m.predict(Xval))))
    return np.mean(scores)

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('\nBest CV RMSE :', round(study.best_value, 4))
print('Best params  :', study.best_params)

In [ ]:
# ── 15. Optuna Visualisations ──────────────────────────────────────────────────
# Optimization history
history = [(t.number, t.value) for t in study.trials if t.value is not None]
nums, vals = zip(*history)

plt.figure(figsize=(10, 4))
plt.plot(nums, vals, alpha=0.6, marker='o', ms=3, label='Trial RMSE')
best_so_far = pd.Series(vals).cummin()
plt.plot(nums, best_so_far, color='red', lw=2, label='Best so far')
plt.xlabel('Trial')
plt.ylabel('CV RMSE')
plt.title('Optuna — Optimization History')
plt.legend()
plt.tight_layout()
plt.show()

# Parameter importances
importances = optuna.importance.get_param_importances(study)
plt.figure(figsize=(8, 4))
plt.barh(list(importances.keys()), list(importances.values()), color='steelblue')
plt.xlabel('Importance')
plt.title('Hyperparameter Importances (Optuna)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 16. Train Final Tuned Model ────────────────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({'random_state': 42, 'verbose': -1})

model_tuned = lgb.LGBMRegressor(**best_params)
model_tuned.fit(X_train, y_train)
pred_tuned = model_tuned.predict(X_test)

rmse_t = np.sqrt(mean_squared_error(y_test, pred_tuned))
mae_t  = mean_absolute_error(y_test, pred_tuned)
r2_t   = r2_score(y_test, pred_tuned)
mape_t = 100 * np.mean(np.abs(pred_tuned - y_test) / np.where(y_test == 0, 1, y_test))

print('─── Tuned Model ───')
print(f'RMSE : {rmse_t:.3f}')
print(f'MAE  : {mae_t:.3f}')
print(f'MAPE : {mape_t:.2f}%')
print(f'R²   : {r2_t:.4f}')

## Model Comparison

We train **three models** on identical features and data to justify choosing LightGBM with Optuna tuning.

| Model | Description | Expected |
|---|---|---|
| Linear Regression | Simple straight-line model | Fails — demand is non-linear |
| LightGBM (default) | Tree-based, no tuning | Good — captures non-linearity |
| LightGBM (Optuna) | Tree-based, auto-tuned | Best — lowest error |

> **Why LR fails:** At 7am demand is ~130 kW, at 8am it jumps to ~440 kW. 
A straight line cannot capture this sudden spike. 
LightGBM decision trees learn these jumps automatically.


In [ ]:
# ── 17. Model Comparison: Linear Regression → LightGBM baseline → LightGBM tuned ──
from sklearn.linear_model      import LinearRegression
from sklearn.preprocessing     import StandardScaler
from sklearn.pipeline          import Pipeline
from sklearn.metrics           import r2_score, mean_squared_error, mean_absolute_error

# ── 1. Linear Regression ──────────────────────────────────────────────────────
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])
lr_pipe.fit(X_train, y_train)
pred_lr   = lr_pipe.predict(X_test)
rmse_lr   = np.sqrt(mean_squared_error(y_test, pred_lr))
mae_lr    = mean_absolute_error(y_test, pred_lr)
r2_lr     = r2_score(y_test, pred_lr)
mape_lr   = 100 * np.mean(np.abs(pred_lr - y_test) / np.where(y_test == 0, 1, y_test))

# ── 2. Reuse already-trained models ───────────────────────────────────────────
# pred_baseline and pred_tuned are already computed in cells 12 and 15

# ── 3. Full comparison table ──────────────────────────────────────────────────
comparison_full = pd.DataFrame({
    'Model'   : ['Linear Regression', 'LightGBM (default)', 'LightGBM (Optuna)'],
    'R²'      : [round(r2_lr,  4), round(r2_b,   4), round(r2_t,  4)],
    'RMSE'    : [round(rmse_lr,3), round(rmse_b, 3), round(rmse_t,3)],
    'MAE'     : [round(mae_lr, 3), round(mae_b,  3), round(mae_t, 3)],
    'MAPE (%)': [round(mape_lr,2), round(mape_b, 2), round(mape_t,2)],
    'Selected': ['No', 'No', '✓ Yes'],
})

print('=== Model Comparison ===')
print(comparison_full.to_string(index=False))

# ── 4. Comparison charts ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

names_c  = ['Linear\nRegression', 'LightGBM\n(default)', 'LightGBM\n(Optuna)']
r2_c     = [r2_lr,   r2_b,   r2_t]
rmse_c   = [rmse_lr, rmse_b, rmse_t]
colors_c = ['#d0d0d0', '#7fb3f5', '#1a73e8']

# R² bar chart
bars1 = ax1.bar(names_c, r2_c, color=colors_c, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars1, r2_c):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{v}', ha='center', fontsize=11, fontweight='bold')
ax1.axhline(0.95, color='green', linestyle='--', linewidth=1.5, label='Target R² ≥ 0.95')
ax1.set_ylim(0.4, 1.05)
ax1.set_title('R² score — higher is better', fontweight='bold')
ax1.set_ylabel('R²')
ax1.legend(fontsize=9)

# RMSE bar chart
bars2 = ax2.bar(names_c, rmse_c, color=colors_c, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars2, rmse_c):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{v}', ha='center', fontsize=11, fontweight='bold')
ax2.set_title('RMSE — lower is better', fontweight='bold')
ax2.set_ylabel('RMSE (kW)')

plt.suptitle('Model comparison — same data, three approaches',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 5. Conclusion ─────────────────────────────────────────────────────────────
print(f'\nConclusion:')
print(f'  Linear Regression R² = {r2_lr:.4f} — fails because demand is non-linear.')
print(f'  The 8am spike (130→440 kW) cannot be captured by a straight line.')
print(f'  LightGBM baseline R² = {r2_b:.4f} — tree-based model handles non-linearity.')
print(f'  LightGBM Optuna   R² = {r2_t:.4f} — best: tuning reduces RMSE by {rmse_b-rmse_t:.3f} kW.')
print(f'  → LightGBM with Optuna tuning selected as final model.')


In [ ]:
# ── 18. Metrics Comparison Table (updated to include Linear Regression) ──────────
print('=== Full Model Comparison Summary ===')
print(comparison_full.to_string(index=False))


In [ ]:
# ── 19. TimeSeriesSplit Cross-Validation on Tuned Model ────────────────────────

cv_rmse, cv_mae, cv_r2 = [], [], []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    m = lgb.LGBMRegressor(**best_params)
    m.fit(Xtr, ytr)
    preds = m.predict(Xval)

    fold_rmse = np.sqrt(mean_squared_error(yval, preds))
    fold_mae  = mean_absolute_error(yval, preds)
    fold_r2   = r2_score(yval, preds)

    cv_rmse.append(fold_rmse)
    cv_mae.append(fold_mae)
    cv_r2.append(fold_r2)
    print(f'Fold {fold} — RMSE: {fold_rmse:.3f}  MAE: {fold_mae:.3f}  R²: {fold_r2:.4f}')

print(f'\nMean RMSE : {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}')
print(f'Mean MAE  : {np.mean(cv_mae):.3f} ± {np.std(cv_mae):.3f}')
print(f'Mean R²   : {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}')

In [ ]:
# ── 20. CV RMSE per Fold Plot ──────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_rmse, color='steelblue', edgecolor='white')
plt.axhline(np.mean(cv_rmse), color='red', linestyle='--', label=f'Mean = {np.mean(cv_rmse):.3f}')
plt.xlabel('Fold')
plt.ylabel('RMSE')
plt.title('TimeSeriesSplit CV — RMSE per Fold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 21. Actual vs Predicted (Full Test Set) ────────────────────────────────────
plt.figure(figsize=(15, 5))
plt.plot(test.index, y_test.values,      label='Actual',    alpha=0.8)
plt.plot(test.index, pred_tuned, '--',   label='Predicted', alpha=0.8)
plt.title('Actual vs Predicted — Tuned Model (Full Test Set)')
plt.ylabel('kW')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 22. Actual vs Predicted (One Week) ────────────────────────────────────────
one_week = test.iloc[:168]
pred_1w  = pred_tuned[:168]

plt.figure(figsize=(15, 5))
plt.plot(one_week.index, one_week[TARGET].values, label='Actual')
plt.plot(one_week.index, pred_1w, '--',            label='Predicted')
plt.title('Actual vs Predicted — First Week of Test Set')
plt.ylabel('kW')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 23. Residual Analysis ──────────────────────────────────────────────────────
residuals = y_test.values - pred_tuned

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residuals over time
axes[0].plot(test.index, residuals, alpha=0.5, linewidth=0.6)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals Over Time')
axes[0].set_ylabel('Error (kW)')

# Histogram
axes[1].hist(residuals, bins=60, edgecolor='white', color='steelblue')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Error (kW)')

# Predicted vs Actual scatter
axes[2].scatter(pred_tuned, y_test.values, alpha=0.2, s=5, color='steelblue')
lims = [min(pred_tuned.min(), y_test.min()),
        max(pred_tuned.max(), y_test.max())]
axes[2].plot(lims, lims, 'r--')
axes[2].set_title('Predicted vs Actual')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.suptitle('Residual Analysis — Tuned Model', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 24. Feature Importance ─────────────────────────────────────────────────────
lgb.plot_importance(model_tuned, figsize=(10, 6), importance_type='gain',
                    title='Feature Importance (Gain) — Tuned Model')
plt.tight_layout()
plt.show()

In [ ]:
# ── 25. Average Demand Patterns ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

weekday_avg = df_feat.groupby('weekday')['total_demand'].mean()
weekday_avg.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Average Demand by Weekday')
axes[0].set_xlabel('Weekday (0=Mon)')
axes[0].set_ylabel('Avg Demand (kW)')

hourly_avg = df_feat.groupby('hour')['total_demand'].mean()
hourly_avg.plot(ax=axes[1], color='steelblue', marker='o')
axes[1].set_title('Average Demand by Hour')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Avg Demand (kW)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 26. Final Summary ───────────────────────────────────────────────────────
print('=' * 60)
print('              FINAL MODEL SUMMARY')
print('=' * 60)
print('\n── Model Comparison ──')
print(comparison_full.to_string(index=False))
print()
print(f'── TimeSeriesSplit CV (5-fold) on Tuned Model ──')
print(f'  Mean RMSE : {np.mean(cv_rmse):.3f} ± {np.std(cv_rmse):.3f}')
print(f'  Mean MAE  : {np.mean(cv_mae):.3f}  ± {np.std(cv_mae):.3f}')
print(f'  Mean R²   : {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}')
print()
print('── Best Optuna Hyperparameters ──')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')
print()
print('── Final Model Performance (Test Set) ──')
print(f'  R²   : {r2_t:.4f}')
print(f'  RMSE : {rmse_t:.3f} kW')
print(f'  MAE  : {mae_t:.3f} kW')
print(f'  MAPE : {mape_t:.2f}%')
print('=' * 60)


In [ ]:
# ── 27. Save model.pkl for deployment ──────────────────────────────────────────
import pickle

artifact = {
    # The trained model (uses best Optuna params)
    'model'   : model_tuned,

    # Feature list in the exact order the model expects
    'features': FEATURES,

    # Test-set metrics saved alongside — surfaced by the API's GET /
    'metrics' : {
        'rmse': round(rmse_t, 4),
        'mae' : round(mae_t,  4),
        'mape': round(mape_t, 4),
        'r2'  : round(r2_t,   4),
    },
}

with open('model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print('Saved model.pkl successfully!')
print(f'  Features : {FEATURES}')
print(f'  RMSE     : {round(rmse_t, 4)}')
print(f'  R²       : {round(r2_t,  4)}')